# CNN

In [ ]:
# -*- coding: utf-8 -*-
# UCI-HAR (GAP vs TPA vs Gated-TPA) + Robustness (test-only) for EACH model
# - Train/Val: train-only normalization (fit on train subset only)
# - Best model selection per architecture: best VAL macro-F1
# - Robustness: apply perturbation on RAW TEST only -> normalize(train stats) -> eval
# - Logging ONLY (no saving)

import os, random, copy
import numpy as np
from dataclasses import dataclass
from typing import Dict, Tuple, List

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score


# ==============================================================================
# 1) Reproducibility
# ==============================================================================
SEED = 42
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)


# ==============================================================================
# 2) Config
# ==============================================================================
@dataclass
class Config:
    data_dir: str = "/content/drive/MyDrive/Colab Notebooks/HAR/har_orig_datasets/UCI_HAR"

    epochs: int = 100
    batch_size: int = 128
    lr: float = 1e-4
    weight_decay: float = 1e-4
    grad_clip: float = 1.0
    label_smoothing: float = 0.05

    patience: int = 20
    min_delta: float = 1e-4
    val_split: float = 0.2

    d_model: int = 128

    tpa_num_prototypes: int = 16
    tpa_heads: int = 4
    tpa_dropout: float = 0.1
    tpa_temperature: float = 0.07

    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    num_workers: int = 2

cfg = Config()


# ==============================================================================
# 3) UCI-HAR Loader
# ==============================================================================
def load_ucihar_split(data_path: str, split: str) -> Tuple[np.ndarray, np.ndarray]:
    assert split in ["train", "test"]

    if split == "train":
        y = np.loadtxt(os.path.join(data_path, "train", "y_train.txt"))
        signal_path = os.path.join(data_path, "train", "Inertial Signals")
    else:
        y = np.loadtxt(os.path.join(data_path, "test", "y_test.txt"))
        signal_path = os.path.join(data_path, "test", "Inertial Signals")

    signal_files = [
        "total_acc_x", "total_acc_y", "total_acc_z",
        "body_acc_x",  "body_acc_y",  "body_acc_z",
        "body_gyro_x", "body_gyro_y", "body_gyro_z",
    ]

    signals = []
    for sf in signal_files:
        filename = os.path.join(signal_path, f"{sf}_{split}.txt")
        sig = np.loadtxt(filename)  # (N, T)
        signals.append(sig)

    X = np.stack(signals, axis=-1).astype(np.float32)  # (N, T, 9)
    y = (y - 1).astype(np.int64)  # 1..6 -> 0..5
    return X, y


# ==============================================================================
# 4) Train-only Normalization (channel-wise)
# ==============================================================================
def compute_channel_mean_std(X_ntc: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    X = X_ntc.reshape(-1, X_ntc.shape[-1])  # (N*T, C)
    mean_c = X.mean(axis=0).astype(np.float32)
    std_c = (X.std(axis=0) + 1e-8).astype(np.float32)
    return mean_c, std_c

def normalize_ntc(X_ntc: np.ndarray, mean_c: np.ndarray, std_c: np.ndarray) -> np.ndarray:
    return ((X_ntc - mean_c[None, None, :]) / std_c[None, None, :]).astype(np.float32)

class PreloadedDataset(Dataset):
    def __init__(self, X_ntc: np.ndarray, y: np.ndarray):
        super().__init__()
        self.X = torch.from_numpy(X_ntc).float()
        self.y = torch.from_numpy(y).long()
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]


# ==============================================================================
# 5) Models
# ==============================================================================
class ConvBNAct(nn.Module):
    def __init__(self, c_in, c_out, k, s=1, p=None, g=1):
        super().__init__()
        self.c = nn.Conv1d(c_in, c_out, k, s, k//2 if p is None else p, groups=g, bias=False)
        self.bn = nn.BatchNorm1d(c_out)
        self.act = nn.GELU()
    def forward(self, x):
        return self.act(self.bn(self.c(x)))

class MultiPathCNN(nn.Module):
    def __init__(self, in_ch=9, d_model=128, branches=(3,5,9,15), stride=2):
        super().__init__()
        h = d_model // 2
        self.pre = ConvBNAct(in_ch, h, 1)
        self.branches = nn.ModuleList([
            nn.Sequential(ConvBNAct(h, h, k, stride, g=h), ConvBNAct(h, h, 1))
            for k in branches
        ])
        self.post = ConvBNAct(len(branches)*h, d_model, 1)

    def forward(self, x_bct):
        z = self.pre(x_bct)
        zs = [b(z) for b in self.branches]
        return self.post(torch.cat(zs, dim=1))  # (B, D, T')

class GAPModel(nn.Module):
    def __init__(self, d_model=128, num_classes=6, in_ch=9):
        super().__init__()
        self.backbone = MultiPathCNN(in_ch=in_ch, d_model=d_model)
        self.fc = nn.Linear(d_model, num_classes)
    def forward(self, x_btc):
        x = x_btc.transpose(1, 2)
        fmap = self.backbone(x)
        feat = fmap.transpose(1, 2)
        pooled = feat.mean(dim=1)
        return self.fc(pooled)

class ProductionTPA(nn.Module):
    def __init__(self, dim, num_prototypes=16, heads=4, dropout=0.1, temperature=0.07):
        super().__init__()
        assert dim % heads == 0
        self.heads = heads
        self.head_dim = dim // heads
        self.num_prototypes = num_prototypes
        self.temperature = temperature

        self.proto = nn.Parameter(torch.randn(num_prototypes, dim) * 0.02)
        self.pre_norm = nn.LayerNorm(dim)

        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim, bias=False)

        self.fuse = nn.Sequential(
            nn.Linear(dim, dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x_btd):
        B, T, D = x_btd.shape
        P = self.num_prototypes

        x = self.pre_norm(x_btd)
        K = self.k_proj(x)
        V = self.v_proj(x)
        Qp = self.q_proj(self.proto).unsqueeze(0).expand(B, -1, -1)

        def split_heads(t, L):
            return t.view(B, L, self.heads, self.head_dim).transpose(1, 2)

        Qh = split_heads(Qp, P)
        Kh = split_heads(K, T)
        Vh = split_heads(V, T)

        Qh = F.normalize(Qh, dim=-1)
        Kh = F.normalize(Kh, dim=-1)

        scores = torch.matmul(Qh, Kh.transpose(-2, -1)) / self.temperature
        attn = F.softmax(scores, dim=-1)
        attn = torch.nan_to_num(attn, nan=0.0)
        attn = self.dropout(attn)

        proto_tokens = torch.matmul(attn, Vh)  # (B,H,P,dh)
        proto_tokens = proto_tokens.transpose(1, 2).contiguous().view(B, P, D)
        z_tpa = proto_tokens.mean(dim=1)
        return self.fuse(z_tpa)

class TPAModel(nn.Module):
    def __init__(self, d_model=128, num_classes=6, in_ch=9, tpa_config=None):
        super().__init__()
        self.backbone = MultiPathCNN(in_ch=in_ch, d_model=d_model)
        self.tpa = ProductionTPA(
            dim=d_model,
            num_prototypes=tpa_config["num_prototypes"],
            heads=tpa_config["heads"],
            dropout=tpa_config["dropout"],
            temperature=tpa_config["temperature"],
        )
        self.classifier = nn.Linear(d_model, num_classes)
    def forward(self, x_btc):
        x = x_btc.transpose(1, 2)
        fmap = self.backbone(x)
        feat = fmap.transpose(1, 2)
        z = self.tpa(feat)
        return self.classifier(z)

class GatedTPAModel(nn.Module):
    def __init__(self, d_model=128, num_classes=6, in_ch=9, tpa_config=None):
        super().__init__()
        self.backbone = MultiPathCNN(in_ch=in_ch, d_model=d_model)
        self.tpa = ProductionTPA(
            dim=d_model,
            num_prototypes=tpa_config["num_prototypes"],
            heads=tpa_config["heads"],
            dropout=tpa_config["dropout"],
            temperature=tpa_config["temperature"],
        )
        self.gate = nn.Sequential(nn.Linear(d_model * 2, d_model), nn.Sigmoid())
        self.classifier = nn.Linear(d_model, num_classes)
    def forward(self, x_btc):
        x = x_btc.transpose(1, 2)
        fmap = self.backbone(x)
        feat = fmap.transpose(1, 2)

        z_gap = feat.mean(dim=1)
        z_tpa = self.tpa(feat)

        g = self.gate(torch.cat([z_gap, z_tpa], dim=-1))
        z = g * z_gap + (1.0 - g) * z_tpa
        return self.classifier(z)

def create_model(model_name: str, cfg: Config):
    tpa_config = dict(
        num_prototypes=cfg.tpa_num_prototypes,
        heads=cfg.tpa_heads,
        dropout=cfg.tpa_dropout,
        temperature=cfg.tpa_temperature,
    )
    if model_name == "GAP":
        return GAPModel(d_model=cfg.d_model, num_classes=6, in_ch=9).to(cfg.device)
    if model_name == "TPA":
        return TPAModel(d_model=cfg.d_model, num_classes=6, in_ch=9, tpa_config=tpa_config).to(cfg.device)
    if model_name == "Gated-TPA":
        return GatedTPAModel(d_model=cfg.d_model, num_classes=6, in_ch=9, tpa_config=tpa_config).to(cfg.device)
    raise ValueError(f"Unknown model: {model_name}")


# ==============================================================================
# 6) Train / Eval (best by VAL macro-F1)
# ==============================================================================
def train_one_epoch(model, loader, opt, cfg: Config):
    model.train()
    for x, y in loader:
        x = x.to(cfg.device).float()
        y = y.to(cfg.device)
        opt.zero_grad(set_to_none=True)
        logits = model(x)
        loss = F.cross_entropy(logits, y, label_smoothing=cfg.label_smoothing)
        if torch.isnan(loss):
            continue
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        opt.step()

@torch.no_grad()
def evaluate(model, loader, cfg: Config) -> Tuple[float, float]:
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        x = x.to(cfg.device).float()
        y = y.to(cfg.device)
        logits = model(x)
        ps.append(logits.argmax(dim=-1).cpu().numpy())
        ys.append(y.cpu().numpy())
    y_true = np.concatenate(ys)
    y_pred = np.concatenate(ps)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    return acc, f1

def train_model(model, train_loader, val_loader, cfg: Config) -> Dict:
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    best_val_f1 = -1.0
    best_wts = None
    patience_counter = 0

    for _ep in range(1, cfg.epochs + 1):
        train_one_epoch(model, train_loader, opt, cfg)
        _, val_f1 = evaluate(model, val_loader, cfg)

        if val_f1 > best_val_f1 + cfg.min_delta:
            best_val_f1 = val_f1
            best_wts = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= cfg.patience:
            break

    if best_wts is not None:
        model.load_state_dict(best_wts)

    return {"best_val_f1": float(best_val_f1), "best_state_dict": best_wts}


# ==============================================================================
# 7) Robustness Perturbations (RAW space)
# ==============================================================================
def tempo_shift_one(x_ct: np.ndarray, speed: float) -> np.ndarray:
    C, T = x_ct.shape
    Tp = max(4, int(round(T / speed)))

    t_old = np.linspace(0, 1, T, dtype=np.float32)
    t_new = np.linspace(0, 1, Tp, dtype=np.float32)

    x_tp = np.zeros((C, Tp), dtype=np.float32)
    for c in range(C):
        x_tp[c] = np.interp(t_new, t_old, x_ct[c]).astype(np.float32)

    t_restore = np.linspace(0, 1, T, dtype=np.float32)
    x_out = np.zeros((C, T), dtype=np.float32)
    for c in range(C):
        x_out[c] = np.interp(t_restore, t_new, x_tp[c]).astype(np.float32)

    return x_out

def apply_tempo_shift_raw(X_raw_nct: np.ndarray, speed: float) -> np.ndarray:
    N = X_raw_nct.shape[0]
    out = np.empty_like(X_raw_nct, dtype=np.float32)
    for i in range(N):
        out[i] = tempo_shift_one(X_raw_nct[i], speed)
    return out

def apply_sensor_noise_raw(X_raw_nct: np.ndarray, mode: str, level: float, rng: np.random.Generator) -> np.ndarray:
    Xn = X_raw_nct.astype(np.float32).copy()
    sigma = Xn.transpose(1, 0, 2).reshape(Xn.shape[1], -1).std(axis=1) + 1e-8  # (C,)

    if mode == "gauss":
        noise = rng.normal(0.0, 1.0, size=Xn.shape).astype(np.float32)
        Xn = Xn + noise * (level * sigma[None, :, None])
    elif mode == "bias":
        b = (level * sigma).astype(np.float32)
        Xn = Xn + b[None, :, None]
    elif mode == "scale":
        Xn = Xn * (1.0 + float(level))
    else:
        raise ValueError(f"Unknown sensor noise mode: {mode}")
    return Xn


# ==============================================================================
# 8) Experiment: Train 3 models, then Robustness for EACH model (based on its clean F1)
# ==============================================================================
def run_ucihar_experiment(cfg: Config):
    # ---- Load RAW ----
    X_train_raw, y_train = load_ucihar_split(cfg.data_dir, "train")
    X_test_raw,  y_test  = load_ucihar_split(cfg.data_dir, "test")

    # ---- Train/Val split ----
    idx = np.arange(len(X_train_raw))
    tr_idx, va_idx = train_test_split(idx, test_size=cfg.val_split, random_state=SEED, stratify=y_train)

    # ---- Train-only normalization stats ----
    mean_c, std_c = compute_channel_mean_std(X_train_raw[tr_idx])

    # ---- Normalize clean splits ----
    Xtr = normalize_ntc(X_train_raw[tr_idx], mean_c, std_c)
    ytr = y_train[tr_idx]
    Xva = normalize_ntc(X_train_raw[va_idx], mean_c, std_c)
    yva = y_train[va_idx]
    Xte = normalize_ntc(X_test_raw, mean_c, std_c)
    yte = y_test

    # ---- Loaders ----
    pin = (cfg.device == "cuda")
    g = torch.Generator(device="cpu").manual_seed(SEED)

    train_loader = DataLoader(PreloadedDataset(Xtr, ytr), batch_size=cfg.batch_size, shuffle=True,
                              num_workers=cfg.num_workers, generator=g, pin_memory=pin)
    val_loader   = DataLoader(PreloadedDataset(Xva, yva), batch_size=cfg.batch_size, shuffle=False,
                              num_workers=cfg.num_workers, pin_memory=pin)
    test_loader  = DataLoader(PreloadedDataset(Xte, yte), batch_size=cfg.batch_size, shuffle=False,
                              num_workers=cfg.num_workers, pin_memory=pin)

    # ---- Robust settings (your exact list) ----
    tempo_speeds = [0.85, 0.90, 0.95, 1.05, 1.10, 1.15]
    gauss_levels = [0.05, 0.10, 0.20]
    bias_levels  = [0.05, 0.10, 0.20]
    scale_levels = [0.05, -0.05, 0.10, -0.10, 0.20, -0.20]

    # raw test as (N,C,T)
    X_test_nct = np.transpose(X_test_raw, (0, 2, 1)).astype(np.float32)
    rng = np.random.default_rng(SEED)

    def eval_model_on_perturbed(model, Xpert_nct: np.ndarray) -> Tuple[float, float]:
        Xpert_ntc = np.transpose(Xpert_nct, (0, 2, 1)).astype(np.float32)
        Xnorm = normalize_ntc(Xpert_ntc, mean_c, std_c)
        loader = DataLoader(PreloadedDataset(Xnorm, y_test),
                            batch_size=cfg.batch_size, shuffle=False,
                            num_workers=cfg.num_workers, pin_memory=pin)
        return evaluate(model, loader, cfg)

    # ---- Train + Robust per model ----
    model_names = ["GAP", "TPA", "Gated-TPA"]

    print("\n" + "="*80)
    print("UCI-HAR | CLEAN + ROBUSTNESS (PER MODEL)")
    print("="*80)

    for mn in model_names:
        set_seed(SEED)
        model = create_model(mn, cfg)

        train_info = train_model(model, train_loader, val_loader, cfg)
        clean_acc, clean_f1 = evaluate(model, test_loader, cfg)

        print("\n" + "-"*80)
        print(f"[{mn}]  Clean Test: Acc={clean_acc:.4f} | F1={clean_f1:.4f} | (Val best F1={train_info['best_val_f1']:.4f})")
        print("-"*80)

        # 1) Tempo
        print("[1) Temporal Scaling]")
        for sp in tempo_speeds:
            Xp = apply_tempo_shift_raw(X_test_nct, sp)
            acc, f1 = eval_model_on_perturbed(model, Xp)
            df1 = clean_f1 - f1
            print(f"speed={sp:>4.2f} | Acc={acc:.4f} | F1={f1:.4f} | ΔF1={df1:+.4f}")

        # 2) Gauss
        print("\n[2) Additive Gaussian Noise]")
        for lv in gauss_levels:
            Xp = apply_sensor_noise_raw(X_test_nct, mode="gauss", level=lv, rng=rng)
            acc, f1 = eval_model_on_perturbed(model, Xp)
            df1 = clean_f1 - f1
            print(f"σ={lv:>4.2f} | Acc={acc:.4f} | F1={f1:.4f} | ΔF1={df1:+.4f}")

        # 3) Bias
        print("\n[3) Additive Bias Drift]")
        for lv in bias_levels:
            Xp = apply_sensor_noise_raw(X_test_nct, mode="bias", level=lv, rng=rng)
            acc, f1 = eval_model_on_perturbed(model, Xp)
            df1 = clean_f1 - f1
            print(f"δ={lv:>4.2f} | Acc={acc:.4f} | F1={f1:.4f} | ΔF1={df1:+.4f}")

        # 4) Scale
        print("\n[4) Multiplicative Scale Drift]")
        for lv in scale_levels:
            Xp = apply_sensor_noise_raw(X_test_nct, mode="scale", level=lv, rng=rng)
            acc, f1 = eval_model_on_perturbed(model, Xp)
            df1 = clean_f1 - f1
            sign = "+" if lv > 0 else ""
            print(f"s={sign}{lv:>4.2f} | Acc={acc:.4f} | F1={f1:.4f} | ΔF1={df1:+.4f}")

    print("\n" + "="*80)
    print("DONE")
    print("="*80)


# ==============================================================================
# 9) Run
# ==============================================================================
if __name__ == "__main__":
    run_ucihar_experiment(cfg)


UCI-HAR | CLEAN + ROBUSTNESS (PER MODEL)

--------------------------------------------------------------------------------
[GAP]  Clean Test: Acc=0.9182 | F1=0.9193 | (Val best F1=0.9655)
--------------------------------------------------------------------------------
[1) Temporal Scaling]
speed=0.85 | Acc=0.9192 | F1=0.9204 | ΔF1=-0.0011
speed=0.90 | Acc=0.9189 | F1=0.9200 | ΔF1=-0.0007
speed=0.95 | Acc=0.9189 | F1=0.9200 | ΔF1=-0.0007
speed=1.05 | Acc=0.9189 | F1=0.9200 | ΔF1=-0.0007
speed=1.10 | Acc=0.9189 | F1=0.9200 | ΔF1=-0.0007
speed=1.15 | Acc=0.9189 | F1=0.9200 | ΔF1=-0.0007

[2) Additive Gaussian Noise]
σ=0.05 | Acc=0.9189 | F1=0.9199 | ΔF1=-0.0007
σ=0.10 | Acc=0.9192 | F1=0.9202 | ΔF1=-0.0009
σ=0.20 | Acc=0.9206 | F1=0.9215 | ΔF1=-0.0022

[3) Additive Bias Drift]
δ=0.05 | Acc=0.9121 | F1=0.9136 | ΔF1=+0.0057
δ=0.10 | Acc=0.9182 | F1=0.9195 | ΔF1=-0.0002
δ=0.20 | Acc=0.9165 | F1=0.9174 | ΔF1=+0.0018

[4) Multiplicative Scale Drift]
s=+0.05 | Acc=0.9199 | F1=0.9211 | ΔF1=-0.0

# BiLSTM

In [ ]:
# -*- coding: utf-8 -*-
# UCI-HAR (GAP vs TPA vs Gated-TPA) + Robustness (test-only) for EACH model
# - Train/Val: train-only normalization (fit on train subset only)
# - Best model selection per architecture: best VAL macro-F1
# - Robustness: apply perturbation on RAW TEST only -> normalize(train stats) -> eval
# - Logging ONLY (no saving)
#
# IMPORTANT:
#   - Model code (BiLSTMBackbone/GAP/TPA/Gated-TPA/ProductionTPA) is kept as-is in structure.
#   - Only minimal wiring changes for UCI-HAR: in_ch=9, num_classes=6, and create_model signature match.
#   - Input to model is (B, T, C) from UCI-HAR loader -> BiLSTM uses batch_first=True so OK.

import os, random, copy
import numpy as np
from dataclasses import dataclass
from typing import Dict, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score


# ==============================================================================
# 1) Reproducibility
# ==============================================================================
SEED = 42
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)


# ==============================================================================
# 2) Config
# ==============================================================================
@dataclass
class Config:
    data_dir: str = "/content/drive/MyDrive/Colab Notebooks/HAR/har_orig_datasets/UCI_HAR"

    epochs: int = 100
    batch_size: int = 128
    lr: float = 1e-4
    weight_decay: float = 1e-4
    grad_clip: float = 1.0
    label_smoothing: float = 0.05

    patience: int = 20
    min_delta: float = 1e-4
    val_split: float = 0.2

    d_model: int = 128

    tpa_num_prototypes: int = 16
    tpa_heads: int = 4
    tpa_dropout: float = 0.1
    tpa_temperature: float = 0.07

    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    num_workers: int = 2

cfg = Config()


# ==============================================================================
# 3) UCI-HAR Loader
# ==============================================================================
def load_ucihar_split(data_path: str, split: str) -> Tuple[np.ndarray, np.ndarray]:
    assert split in ["train", "test"]

    if split == "train":
        y = np.loadtxt(os.path.join(data_path, "train", "y_train.txt"))
        signal_path = os.path.join(data_path, "train", "Inertial Signals")
    else:
        y = np.loadtxt(os.path.join(data_path, "test", "y_test.txt"))
        signal_path = os.path.join(data_path, "test", "Inertial Signals")

    signal_files = [
        "total_acc_x", "total_acc_y", "total_acc_z",
        "body_acc_x",  "body_acc_y",  "body_acc_z",
        "body_gyro_x", "body_gyro_y", "body_gyro_z",
    ]

    signals = []
    for sf in signal_files:
        filename = os.path.join(signal_path, f"{sf}_{split}.txt")
        sig = np.loadtxt(filename)  # (N, T)
        signals.append(sig)

    X = np.stack(signals, axis=-1).astype(np.float32)  # (N, T, 9)
    y = (y - 1).astype(np.int64)  # 1..6 -> 0..5
    return X, y


# ==============================================================================
# 4) Train-only Normalization (channel-wise)
# ==============================================================================
def compute_channel_mean_std(X_ntc: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    X = X_ntc.reshape(-1, X_ntc.shape[-1])  # (N*T, C)
    mean_c = X.mean(axis=0).astype(np.float32)
    std_c = (X.std(axis=0) + 1e-8).astype(np.float32)
    return mean_c, std_c

def normalize_ntc(X_ntc: np.ndarray, mean_c: np.ndarray, std_c: np.ndarray) -> np.ndarray:
    return ((X_ntc - mean_c[None, None, :]) / std_c[None, None, :]).astype(np.float32)

class PreloadedDataset(Dataset):
    def __init__(self, X_ntc: np.ndarray, y: np.ndarray):
        super().__init__()
        self.X = torch.from_numpy(X_ntc).float()
        self.y = torch.from_numpy(y).long()
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]


# ==============================================================================
# 5) Models (KEEP AS-IS, only minimal UCI-HAR wiring: in_ch=9, num_classes=6)
# ==============================================================================
# ========================
# Model Components
# ========================
class BiLSTMBackbone(nn.Module):
    """BiLSTM backbone for all models
    Args:
        in_ch: input channels (UCI-HAR: 9)
        d_model: output dimension (default: 128)
        hidden_dim: LSTM hidden dimension (default: 64, bidirectional -> 128)
        num_layers: number of LSTM layers (default: 2)
        dropout: dropout rate (default: 0.1)
    """
    def __init__(self, in_ch=27, d_model=128, hidden_dim=64, num_layers=2, dropout=0.1):
        super().__init__()
        self.d_model = d_model

        # BiLSTM: hidden_dim=64 → 양방향이므로 출력은 128
        self.lstm = nn.LSTM(
            input_size=in_ch,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        # 투영층: BiLSTM 출력(128) → d_model(128)
        self.projection = nn.Linear(hidden_dim * 2, d_model)

        # 정규화 및 드롭아웃
        self.layer_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # NOTE: For UCI-HAR loader, x comes as [B, T, C] already.
        # x: [B, C, T] → [B, T, C] (LSTM은 [B, T, C] 형태 입력)
        # x = x.transpose(1, 2)

        # BiLSTM
        lstm_out, _ = self.lstm(x)  # [B, T, hidden_dim*2=128]

        # 투영
        out = self.projection(lstm_out)  # [B, T, d_model=128]

        # 정규화 + 드롭아웃
        out = self.layer_norm(out)
        out = self.dropout(out)

        # [B, T, D] → [B, D, T] (기존 코드와의 호환성)
        return out.transpose(1, 2)

# ========================
# GAP Model
# ========================
class GAPModel(nn.Module):
    """Baseline: Global Average Pooling"""
    def __init__(self, d_model=128, num_classes=12):
        super().__init__()
        self.backbone = BiLSTMBackbone(d_model=d_model)
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x):
        fmap = self.backbone(x)  # [B, D, T]
        features = fmap.transpose(1, 2)  # [B, T, D]
        pooled = features.mean(dim=1)  # [B, D]
        logits = self.fc(pooled)
        return logits

# ========================
# Pure-TPA
# ========================
class ProductionTPA(nn.Module):
    """Pure TPA"""
    def __init__(self, dim, num_prototypes=16, heads=4, dropout=0.1,
                 temperature=0.07):
        super().__init__()
        assert dim % heads == 0

        self.dim = dim
        self.heads = heads
        self.head_dim = dim // heads
        self.num_prototypes = num_prototypes
        self.temperature = temperature

        self.proto = nn.Parameter(torch.randn(num_prototypes, dim) * 0.02)

        self.pre_norm = nn.LayerNorm(dim)

        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim, bias=False)

        self.fuse = nn.Sequential(
            nn.Linear(dim, dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim)
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, D = x.shape
        P = self.num_prototypes

        x_norm = self.pre_norm(x)

        K = self.k_proj(x_norm)
        V = self.v_proj(x_norm)
        Qp = self.q_proj(self.proto).unsqueeze(0).expand(B, -1, -1)

        def split_heads(t, length):
            return t.view(B, length, self.heads, self.head_dim).transpose(1, 2)

        Qh = split_heads(Qp, P)
        Kh = split_heads(K, T)
        Vh = split_heads(V, T)

        Qh = F.normalize(Qh, dim=-1)
        Kh = F.normalize(Kh, dim=-1)

        scores = torch.matmul(Qh, Kh.transpose(-2, -1)) / self.temperature
        attn = F.softmax(scores, dim=-1)
        attn = torch.nan_to_num(attn, nan=0.0)
        attn = self.dropout(attn)

        proto_tokens = torch.matmul(attn, Vh)
        proto_tokens = proto_tokens.transpose(1, 2).contiguous().view(B, P, D)

        z_tpa = proto_tokens.mean(dim=1)

        z = self.fuse(z_tpa)

        return z

class TPAModel(nn.Module):
    def __init__(self, d_model=128, num_classes=12, tpa_config=None):
        super().__init__()
        self.backbone = BiLSTMBackbone(d_model=d_model)
        self.tpa = ProductionTPA(
            dim=d_model,
            num_prototypes=tpa_config['num_prototypes'],
            heads=tpa_config['heads'],
            dropout=tpa_config['dropout'],
            temperature=tpa_config['temperature']
        )
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x):
        fmap = self.backbone(x)  # [B, D, T]
        features = fmap.transpose(1, 2)  # [B, T, D]
        z = self.tpa(features)  # [B, D]
        logits = self.classifier(z)
        return logits

# ========================
# Gated-TPA
# ========================
class GatedTPAModel(nn.Module):
    def __init__(self, d_model=128, num_classes=12, tpa_config=None):
        super().__init__()
        self.backbone = BiLSTMBackbone(d_model=d_model)
        self.tpa = ProductionTPA(
            dim=d_model,
            num_prototypes=tpa_config['num_prototypes'],
            heads=tpa_config['heads'],
            dropout=tpa_config['dropout'],
            temperature=tpa_config['temperature']
        )

        # Gating mechanism
        self.gate = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.Sigmoid()
        )

        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x):
        fmap = self.backbone(x)  # [B, D, T]
        features = fmap.transpose(1, 2)  # [B, T, D]

        # GAP branch
        z_gap = features.mean(dim=1)  # [B, D]

        # TPA branch
        z_tpa = self.tpa(features)  # [B, D]

        # Gating
        gate_input = torch.cat([z_gap, z_tpa], dim=-1)
        g = self.gate(gate_input)

        # Gated fusion
        z = g * z_gap + (1 - g) * z_tpa

        logits = self.classifier(z)
        return logits

# ---- Minimal wiring for UCI-HAR (in_ch=9, num_classes=6) without changing model structure ----
def create_model(model_name: str, cfg: Config):
    tpa_config = dict(
        num_prototypes=cfg.tpa_num_prototypes,
        heads=cfg.tpa_heads,
        dropout=cfg.tpa_dropout,
        temperature=cfg.tpa_temperature,
    )

    if model_name == "GAP":
        model = GAPModel(d_model=cfg.d_model, num_classes=6).to(cfg.device)
        # keep structure; just set backbone input channels for UCI-HAR
        model.backbone = BiLSTMBackbone(in_ch=9, d_model=cfg.d_model).to(cfg.device)
        return model

    if model_name == "TPA":
        model = TPAModel(d_model=cfg.d_model, num_classes=6, tpa_config=tpa_config).to(cfg.device)
        model.backbone = BiLSTMBackbone(in_ch=9, d_model=cfg.d_model).to(cfg.device)
        return model

    if model_name == "Gated-TPA":
        model = GatedTPAModel(d_model=cfg.d_model, num_classes=6, tpa_config=tpa_config).to(cfg.device)
        model.backbone = BiLSTMBackbone(in_ch=9, d_model=cfg.d_model).to(cfg.device)
        return model

    raise ValueError(f"Unknown model: {model_name}")


# ==============================================================================
# 6) Train / Eval (best by VAL macro-F1)
# ==============================================================================
def train_one_epoch(model, loader, opt, cfg: Config):
    model.train()
    for x, y in loader:
        x = x.to(cfg.device).float()
        y = y.to(cfg.device)
        opt.zero_grad(set_to_none=True)
        logits = model(x)
        loss = F.cross_entropy(logits, y, label_smoothing=cfg.label_smoothing)
        if torch.isnan(loss):
            continue
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        opt.step()

@torch.no_grad()
def evaluate(model, loader, cfg: Config) -> Tuple[float, float]:
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        x = x.to(cfg.device).float()
        y = y.to(cfg.device)
        logits = model(x)
        ps.append(logits.argmax(dim=-1).cpu().numpy())
        ys.append(y.cpu().numpy())
    y_true = np.concatenate(ys)
    y_pred = np.concatenate(ps)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    return acc, f1

def train_model(model, train_loader, val_loader, cfg: Config) -> Dict:
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    best_val_f1 = -1.0
    best_wts = None
    patience_counter = 0

    for _ep in range(1, cfg.epochs + 1):
        train_one_epoch(model, train_loader, opt, cfg)
        _, val_f1 = evaluate(model, val_loader, cfg)

        if val_f1 > best_val_f1 + cfg.min_delta:
            best_val_f1 = val_f1
            best_wts = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= cfg.patience:
            break

    if best_wts is not None:
        model.load_state_dict(best_wts)

    return {"best_val_f1": float(best_val_f1), "best_state_dict": best_wts}


# ==============================================================================
# 7) Robustness Perturbations (RAW space)
# ==============================================================================
def tempo_shift_one(x_ct: np.ndarray, speed: float) -> np.ndarray:
    C, T = x_ct.shape
    Tp = max(4, int(round(T / speed)))

    t_old = np.linspace(0, 1, T, dtype=np.float32)
    t_new = np.linspace(0, 1, Tp, dtype=np.float32)

    x_tp = np.zeros((C, Tp), dtype=np.float32)
    for c in range(C):
        x_tp[c] = np.interp(t_new, t_old, x_ct[c]).astype(np.float32)

    t_restore = np.linspace(0, 1, T, dtype=np.float32)
    x_out = np.zeros((C, T), dtype=np.float32)
    for c in range(C):
        x_out[c] = np.interp(t_restore, t_new, x_tp[c]).astype(np.float32)

    return x_out

def apply_tempo_shift_raw(X_raw_nct: np.ndarray, speed: float) -> np.ndarray:
    N = X_raw_nct.shape[0]
    out = np.empty_like(X_raw_nct, dtype=np.float32)
    for i in range(N):
        out[i] = tempo_shift_one(X_raw_nct[i], speed)
    return out

def apply_sensor_noise_raw(X_raw_nct: np.ndarray, mode: str, level: float, rng: np.random.Generator) -> np.ndarray:
    Xn = X_raw_nct.astype(np.float32).copy()
    sigma = Xn.transpose(1, 0, 2).reshape(Xn.shape[1], -1).std(axis=1) + 1e-8  # (C,)

    if mode == "gauss":
        noise = rng.normal(0.0, 1.0, size=Xn.shape).astype(np.float32)
        Xn = Xn + noise * (level * sigma[None, :, None])
    elif mode == "bias":
        b = (level * sigma).astype(np.float32)
        Xn = Xn + b[None, :, None]
    elif mode == "scale":
        Xn = Xn * (1.0 + float(level))
    else:
        raise ValueError(f"Unknown sensor noise mode: {mode}")
    return Xn


# ==============================================================================
# 8) Experiment: Train 3 models, then Robustness for EACH model (based on its clean F1)
# ==============================================================================
def run_ucihar_experiment(cfg: Config):
    # ---- Load RAW ----
    X_train_raw, y_train = load_ucihar_split(cfg.data_dir, "train")
    X_test_raw,  y_test  = load_ucihar_split(cfg.data_dir, "test")

    # ---- Train/Val split ----
    idx = np.arange(len(X_train_raw))
    tr_idx, va_idx = train_test_split(idx, test_size=cfg.val_split, random_state=SEED, stratify=y_train)

    # ---- Train-only normalization stats ----
    mean_c, std_c = compute_channel_mean_std(X_train_raw[tr_idx])

    # ---- Normalize clean splits ----
    Xtr = normalize_ntc(X_train_raw[tr_idx], mean_c, std_c)
    ytr = y_train[tr_idx]
    Xva = normalize_ntc(X_train_raw[va_idx], mean_c, std_c)
    yva = y_train[va_idx]
    Xte = normalize_ntc(X_test_raw, mean_c, std_c)
    yte = y_test

    # ---- Loaders ----
    pin = (cfg.device == "cuda")
    g = torch.Generator(device="cpu").manual_seed(SEED)

    train_loader = DataLoader(PreloadedDataset(Xtr, ytr), batch_size=cfg.batch_size, shuffle=True,
                              num_workers=cfg.num_workers, generator=g, pin_memory=pin)
    val_loader   = DataLoader(PreloadedDataset(Xva, yva), batch_size=cfg.batch_size, shuffle=False,
                              num_workers=cfg.num_workers, pin_memory=pin)
    test_loader  = DataLoader(PreloadedDataset(Xte, yte), batch_size=cfg.batch_size, shuffle=False,
                              num_workers=cfg.num_workers, pin_memory=pin)

    # ---- Robust settings (your exact list) ----
    tempo_speeds = [0.85, 0.90, 0.95, 1.05, 1.10, 1.15]
    gauss_levels = [0.05, 0.10, 0.20]
    bias_levels  = [0.05, 0.10, 0.20]
    scale_levels = [0.05, -0.05, 0.10, -0.10, 0.20, -0.20]

    # raw test as (N,C,T)
    X_test_nct = np.transpose(X_test_raw, (0, 2, 1)).astype(np.float32)
    rng = np.random.default_rng(SEED)

    def eval_model_on_perturbed(model, Xpert_nct: np.ndarray) -> Tuple[float, float]:
        Xpert_ntc = np.transpose(Xpert_nct, (0, 2, 1)).astype(np.float32)
        Xnorm = normalize_ntc(Xpert_ntc, mean_c, std_c)
        loader = DataLoader(PreloadedDataset(Xnorm, y_test),
                            batch_size=cfg.batch_size, shuffle=False,
                            num_workers=cfg.num_workers, pin_memory=pin)
        return evaluate(model, loader, cfg)

    # ---- Train + Robust per model ----
    model_names = ["GAP", "TPA", "Gated-TPA"]

    print("\n" + "="*80)
    print("UCI-HAR | CLEAN + ROBUSTNESS (PER MODEL)  |  Backbone: BiLSTM")
    print("="*80)

    for mn in model_names:
        set_seed(SEED)
        model = create_model(mn, cfg)

        train_info = train_model(model, train_loader, val_loader, cfg)
        clean_acc, clean_f1 = evaluate(model, test_loader, cfg)

        print("\n" + "-"*80)
        print(f"[{mn}]  Clean Test: Acc={clean_acc:.4f} | F1={clean_f1:.4f} | (Val best F1={train_info['best_val_f1']:.4f})")
        print("-"*80)

        # 1) Tempo
        print("[1) Temporal Scaling]")
        for sp in tempo_speeds:
            Xp = apply_tempo_shift_raw(X_test_nct, sp)
            acc, f1 = eval_model_on_perturbed(model, Xp)
            df1 = clean_f1 - f1
            print(f"speed={sp:>4.2f} | Acc={acc:.4f} | F1={f1:.4f} | ΔF1={df1:+.4f}")

        # 2) Gauss
        print("\n[2) Additive Gaussian Noise]")
        for lv in gauss_levels:
            Xp = apply_sensor_noise_raw(X_test_nct, mode="gauss", level=lv, rng=rng)
            acc, f1 = eval_model_on_perturbed(model, Xp)
            df1 = clean_f1 - f1
            print(f"σ={lv:>4.2f} | Acc={acc:.4f} | F1={f1:.4f} | ΔF1={df1:+.4f}")

        # 3) Bias
        print("\n[3) Additive Bias Drift]")
        for lv in bias_levels:
            Xp = apply_sensor_noise_raw(X_test_nct, mode="bias", level=lv, rng=rng)
            acc, f1 = eval_model_on_perturbed(model, Xp)
            df1 = clean_f1 - f1
            print(f"δ={lv:>4.2f} | Acc={acc:.4f} | F1={f1:.4f} | ΔF1={df1:+.4f}")

        # 4) Scale
        print("\n[4) Multiplicative Scale Drift]")
        for lv in scale_levels:
            Xp = apply_sensor_noise_raw(X_test_nct, mode="scale", level=lv, rng=rng)
            acc, f1 = eval_model_on_perturbed(model, Xp)
            df1 = clean_f1 - f1
            sign = "+" if lv > 0 else ""
            print(f"s={sign}{lv:>4.2f} | Acc={acc:.4f} | F1={f1:.4f} | ΔF1={df1:+.4f}")

    print("\n" + "="*80)
    print("DONE")
    print("="*80)


# ==============================================================================
# 9) Run
# ==============================================================================
if __name__ == "__main__":
    run_ucihar_experiment(cfg)


UCI-HAR | CLEAN + ROBUSTNESS (PER MODEL)  |  Backbone: BiLSTM

--------------------------------------------------------------------------------
[GAP]  Clean Test: Acc=0.9128 | F1=0.9137 | (Val best F1=0.9662)
--------------------------------------------------------------------------------
[1) Temporal Scaling]
speed=0.85 | Acc=0.9125 | F1=0.9134 | ΔF1=+0.0004
speed=0.90 | Acc=0.9121 | F1=0.9130 | ΔF1=+0.0007
speed=0.95 | Acc=0.9131 | F1=0.9141 | ΔF1=-0.0004
speed=1.05 | Acc=0.9125 | F1=0.9134 | ΔF1=+0.0004
speed=1.10 | Acc=0.9125 | F1=0.9134 | ΔF1=+0.0004
speed=1.15 | Acc=0.9118 | F1=0.9127 | ΔF1=+0.0010

[2) Additive Gaussian Noise]
σ=0.05 | Acc=0.9131 | F1=0.9141 | ΔF1=-0.0004
σ=0.10 | Acc=0.9118 | F1=0.9127 | ΔF1=+0.0010
σ=0.20 | Acc=0.9114 | F1=0.9124 | ΔF1=+0.0013

[3) Additive Bias Drift]
δ=0.05 | Acc=0.9101 | F1=0.9111 | ΔF1=+0.0026
δ=0.10 | Acc=0.9108 | F1=0.9120 | ΔF1=+0.0017
δ=0.20 | Acc=0.9152 | F1=0.9165 | ΔF1=-0.0028

[4) Multiplicative Scale Drift]
s=+0.05 | Acc=0.9101 |

# Transformer

In [ ]:
# -*- coding: utf-8 -*-
# UCI-HAR (GAP vs TPA vs Gated-TPA) + Robustness (test-only) for EACH model
# - Train/Val: train-only normalization (fit on train subset only)
# - Best model selection per architecture: best VAL macro-F1
# - Robustness: apply perturbation on RAW TEST only -> normalize(train stats) -> eval
# - Logging ONLY (no saving)
#
# Transformer backbone version
# - Model code: uses EXACT blocks you pasted (only num_classes/in_channels wired in create_model)

import os, random, copy
import numpy as np
from dataclasses import dataclass
from typing import Dict, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score


# ==============================================================================
# 1) Reproducibility
# ==============================================================================
SEED = 42
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)


# ==============================================================================
# 2) Config
# ==============================================================================
@dataclass
class Config:
    data_dir: str = "/content/drive/MyDrive/Colab Notebooks/HAR/har_orig_datasets/UCI_HAR"

    epochs: int = 100
    batch_size: int = 128
    lr: float = 1e-4
    weight_decay: float = 1e-4
    grad_clip: float = 1.0
    label_smoothing: float = 0.05

    patience: int = 20
    min_delta: float = 1e-4
    val_split: float = 0.2

    # model dims
    d_model: int = 128

    # Transformer hyperparameters (match your pasted code defaults)
    num_layers: int = 2
    n_heads: int = 4
    ff_dim: int = 256
    dropout: float = 0.1
    max_seq_len: int = 200  # UCI-HAR T=128, so 200 is enough

    # TPA hyperparameters
    tpa_num_prototypes: int = 16
    tpa_heads: int = 4
    tpa_dropout: float = 0.1
    tpa_temperature: float = 0.07

    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    num_workers: int = 2

cfg = Config()


# ==============================================================================
# 3) UCI-HAR Loader
# ==============================================================================
def load_ucihar_split(data_path: str, split: str) -> Tuple[np.ndarray, np.ndarray]:
    assert split in ["train", "test"]

    if split == "train":
        y = np.loadtxt(os.path.join(data_path, "train", "y_train.txt"))
        signal_path = os.path.join(data_path, "train", "Inertial Signals")
    else:
        y = np.loadtxt(os.path.join(data_path, "test", "y_test.txt"))
        signal_path = os.path.join(data_path, "test", "Inertial Signals")

    signal_files = [
        "total_acc_x", "total_acc_y", "total_acc_z",
        "body_acc_x",  "body_acc_y",  "body_acc_z",
        "body_gyro_x", "body_gyro_y", "body_gyro_z",
    ]

    signals = []
    for sf in signal_files:
        filename = os.path.join(signal_path, f"{sf}_{split}.txt")
        sig = np.loadtxt(filename)  # (N, T)
        signals.append(sig)

    X = np.stack(signals, axis=-1).astype(np.float32)  # (N, T, 9)
    y = (y - 1).astype(np.int64)  # 1..6 -> 0..5
    return X, y


# ==============================================================================
# 4) Train-only Normalization (channel-wise)
# ==============================================================================
def compute_channel_mean_std(X_ntc: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    X = X_ntc.reshape(-1, X_ntc.shape[-1])  # (N*T, C)
    mean_c = X.mean(axis=0).astype(np.float32)
    std_c = (X.std(axis=0) + 1e-8).astype(np.float32)
    return mean_c, std_c

def normalize_ntc(X_ntc: np.ndarray, mean_c: np.ndarray, std_c: np.ndarray) -> np.ndarray:
    return ((X_ntc - mean_c[None, None, :]) / std_c[None, None, :]).astype(np.float32)

class PreloadedDataset(Dataset):
    def __init__(self, X_ntc: np.ndarray, y: np.ndarray):
        super().__init__()
        self.X = torch.from_numpy(X_ntc).float()  # [N, T, C]
        self.y = torch.from_numpy(y).long()
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]


# ==============================================================================
# 5) Models (EXACT blocks you pasted)
# ==============================================================================
# ========================
# Transformer Backbone Components
# ========================
class PositionalEncoding(nn.Module):
    """Sinusoidal Positional Encoding"""
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)  # [1, max_len, d_model]
        self.register_buffer('pe', pe)

    def forward(self, x):
        """
        Args:
            x: [B, T, D]
        Returns:
            [B, T, D]
        """
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class TransformerBackbone(nn.Module):
    """
    Lightweight Transformer Encoder Backbone
    - 2 layers
    - d_model=128
    - n_heads=4
    - ff_dim=256
    - Dropout=0.1
    """
    def __init__(self,
                 in_channels: int = 27,
                 d_model: int = 128,
                 num_layers: int = 2,
                 n_heads: int = 4,
                 ff_dim: int = 256,
                 dropout: float = 0.1,
                 max_seq_len: int = 200):
        super().__init__()

        self.d_model = d_model

        # Input projection: [B, C, T] -> [B, T, D]
        self.input_projection = nn.Linear(in_channels, d_model)

        # Positional encoding
        self.pos_encoder = PositionalEncoding(d_model, max_seq_len, dropout)

        # Transformer Encoder layers
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True  # Pre-LN for better stability
        )

        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        # Output normalization
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        """
        Args:
            x: [B, C, T] - input sensor data
        Returns:
            [B, T, D] - transformed sequence
        """
        # [B, C, T] -> [B, T, C]
        # x = x.transpose(1, 2)

        # Project to d_model: [B, T, C] -> [B, T, D]
        x = self.input_projection(x)

        # Add positional encoding: [B, T, D]
        x = self.pos_encoder(x)

        # Transformer encoding: [B, T, D]
        x = self.transformer_encoder(x)

        # Final normalization: [B, T, D]
        x = self.norm(x)

        return x

# ========================
# GAP Model with Transformer
# ========================
class GAPModel(nn.Module):
    """Baseline: Global Average Pooling with Transformer Backbone"""
    def __init__(self,
                 in_channels: int = 27,
                 d_model: int = 128,
                 num_layers: int = 2,
                 n_heads: int = 4,
                 ff_dim: int = 256,
                 dropout: float = 0.1,
                 num_classes: int = 12):
        super().__init__()
        self.backbone = TransformerBackbone(
            in_channels=in_channels,
            d_model=d_model,
            num_layers=num_layers,
            n_heads=n_heads,
            ff_dim=ff_dim,
            dropout=dropout
        )
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x):
        features = self.backbone(x)  # [B, T, D]
        pooled = features.mean(dim=1)  # [B, D]
        logits = self.fc(pooled)
        return logits

# ========================
# Pure-TPA with Transformer
# ========================
class ProductionTPA(nn.Module):
    """Pure TPA"""
    def __init__(self, dim, num_prototypes=16, heads=4, dropout=0.1,
                 temperature=0.07):
        super().__init__()
        assert dim % heads == 0

        self.dim = dim
        self.heads = heads
        self.head_dim = dim // heads
        self.num_prototypes = num_prototypes
        self.temperature = temperature

        self.proto = nn.Parameter(torch.randn(num_prototypes, dim) * 0.02)

        self.pre_norm = nn.LayerNorm(dim)

        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim, bias=False)

        self.fuse = nn.Sequential(
            nn.Linear(dim, dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim)
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, D = x.shape
        P = self.num_prototypes

        x_norm = self.pre_norm(x)

        K = self.k_proj(x_norm)
        V = self.v_proj(x_norm)
        Qp = self.q_proj(self.proto).unsqueeze(0).expand(B, -1, -1)

        def split_heads(t, length):
            return t.view(B, length, self.heads, self.head_dim).transpose(1, 2)

        Qh = split_heads(Qp, P)
        Kh = split_heads(K, T)
        Vh = split_heads(V, T)

        Qh = F.normalize(Qh, dim=-1)
        Kh = F.normalize(Kh, dim=-1)

        scores = torch.matmul(Qh, Kh.transpose(-2, -1)) / self.temperature
        attn = F.softmax(scores, dim=-1)
        attn = torch.nan_to_num(attn, nan=0.0)
        attn = self.dropout(attn)

        proto_tokens = torch.matmul(attn, Vh)
        proto_tokens = proto_tokens.transpose(1, 2).contiguous().view(B, P, D)

        z_tpa = proto_tokens.mean(dim=1)

        z = self.fuse(z_tpa)

        return z

class TPAModel(nn.Module):
    def __init__(self,
                 in_channels: int = 27,
                 d_model: int = 128,
                 num_layers: int = 2,
                 n_heads: int = 4,
                 ff_dim: int = 256,
                 dropout: float = 0.1,
                 num_classes: int = 12,
                 tpa_config=None):
        super().__init__()
        self.backbone = TransformerBackbone(
            in_channels=in_channels,
            d_model=d_model,
            num_layers=num_layers,
            n_heads=n_heads,
            ff_dim=ff_dim,
            dropout=dropout
        )
        self.tpa = ProductionTPA(
            dim=d_model,
            num_prototypes=tpa_config['num_prototypes'],
            heads=tpa_config['heads'],
            dropout=tpa_config['dropout'],
            temperature=tpa_config['temperature']
        )
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x):
        features = self.backbone(x)  # [B, T, D]
        z = self.tpa(features)  # [B, D]
        logits = self.classifier(z)
        return logits

# ========================
# Gated-TPA with Transformer
# ========================
class GatedTPAModel(nn.Module):
    """Gated-TPA: Hybrid of TPA and GAP"""
    def __init__(self,
                 in_channels: int = 27,
                 d_model: int = 128,
                 num_layers: int = 2,
                 n_heads: int = 4,
                 ff_dim: int = 256,
                 dropout: float = 0.1,
                 num_classes: int = 12,
                 tpa_config=None):
        super().__init__()
        self.backbone = TransformerBackbone(
            in_channels=in_channels,
            d_model=d_model,
            num_layers=num_layers,
            n_heads=n_heads,
            ff_dim=ff_dim,
            dropout=dropout
        )
        self.tpa = ProductionTPA(
            dim=d_model,
            num_prototypes=tpa_config['num_prototypes'],
            heads=tpa_config['heads'],
            dropout=tpa_config['dropout'],
            temperature=tpa_config['temperature']
        )

        # Gating mechanism
        self.gate = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.Sigmoid()
        )

        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x):
        features = self.backbone(x)  # [B, T, D]

        # TPA path
        z_tpa = self.tpa(features)  # [B, D]

        # GAP path
        z_gap = features.mean(dim=1)  # [B, D]

        # Gating
        gate_input = torch.cat([z_tpa, z_gap], dim=-1)  # [B, 2D]
        alpha = self.gate(gate_input)  # [B, D]

        # Combine
        z = alpha * z_tpa + (1 - alpha) * z_gap  # [B, D]

        logits = self.classifier(z)
        return logits

def create_model(model_name: str, cfg: Config):
    tpa_config = dict(
        num_prototypes=cfg.tpa_num_prototypes,
        heads=cfg.tpa_heads,
        dropout=cfg.tpa_dropout,
        temperature=cfg.tpa_temperature,
    )

    # UCI-HAR: in_channels=9, num_classes=6
    in_ch = 9
    n_cls = 6

    if model_name == "GAP":
        return GAPModel(
            in_channels=in_ch,
            d_model=cfg.d_model,
            num_layers=cfg.num_layers,
            n_heads=cfg.n_heads,
            ff_dim=cfg.ff_dim,
            dropout=cfg.dropout,
            num_classes=n_cls
        ).to(cfg.device)

    if model_name == "TPA":
        return TPAModel(
            in_channels=in_ch,
            d_model=cfg.d_model,
            num_layers=cfg.num_layers,
            n_heads=cfg.n_heads,
            ff_dim=cfg.ff_dim,
            dropout=cfg.dropout,
            num_classes=n_cls,
            tpa_config=tpa_config
        ).to(cfg.device)

    if model_name == "Gated-TPA":
        return GatedTPAModel(
            in_channels=in_ch,
            d_model=cfg.d_model,
            num_layers=cfg.num_layers,
            n_heads=cfg.n_heads,
            ff_dim=cfg.ff_dim,
            dropout=cfg.dropout,
            num_classes=n_cls,
            tpa_config=tpa_config
        ).to(cfg.device)

    raise ValueError(f"Unknown model: {model_name}")


# ==============================================================================
# 6) Train / Eval (best by VAL macro-F1)
# ==============================================================================
def train_one_epoch(model, loader, opt, cfg: Config):
    model.train()
    for x, y in loader:
        x = x.to(cfg.device).float()
        y = y.to(cfg.device)
        opt.zero_grad(set_to_none=True)
        logits = model(x)
        loss = F.cross_entropy(logits, y, label_smoothing=cfg.label_smoothing)
        if torch.isnan(loss):
            continue
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        opt.step()

@torch.no_grad()
def evaluate(model, loader, cfg: Config) -> Tuple[float, float]:
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        x = x.to(cfg.device).float()
        y = y.to(cfg.device)
        logits = model(x)
        ps.append(logits.argmax(dim=-1).cpu().numpy())
        ys.append(y.cpu().numpy())
    y_true = np.concatenate(ys)
    y_pred = np.concatenate(ps)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    return acc, f1

def train_model(model, train_loader, val_loader, cfg: Config) -> Dict:
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    best_val_f1 = -1.0
    best_wts = None
    patience_counter = 0

    for _ep in range(1, cfg.epochs + 1):
        train_one_epoch(model, train_loader, opt, cfg)
        _, val_f1 = evaluate(model, val_loader, cfg)

        if val_f1 > best_val_f1 + cfg.min_delta:
            best_val_f1 = val_f1
            best_wts = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= cfg.patience:
            break

    if best_wts is not None:
        model.load_state_dict(best_wts)

    return {"best_val_f1": float(best_val_f1), "best_state_dict": best_wts}


# ==============================================================================
# 7) Robustness Perturbations (RAW space)
# ==============================================================================
def tempo_shift_one(x_ct: np.ndarray, speed: float) -> np.ndarray:
    C, T = x_ct.shape
    Tp = max(4, int(round(T / speed)))

    t_old = np.linspace(0, 1, T, dtype=np.float32)
    t_new = np.linspace(0, 1, Tp, dtype=np.float32)

    x_tp = np.zeros((C, Tp), dtype=np.float32)
    for c in range(C):
        x_tp[c] = np.interp(t_new, t_old, x_ct[c]).astype(np.float32)

    t_restore = np.linspace(0, 1, T, dtype=np.float32)
    x_out = np.zeros((C, T), dtype=np.float32)
    for c in range(C):
        x_out[c] = np.interp(t_restore, t_new, x_tp[c]).astype(np.float32)

    return x_out

def apply_tempo_shift_raw(X_raw_nct: np.ndarray, speed: float) -> np.ndarray:
    N = X_raw_nct.shape[0]
    out = np.empty_like(X_raw_nct, dtype=np.float32)
    for i in range(N):
        out[i] = tempo_shift_one(X_raw_nct[i], speed)
    return out

def apply_sensor_noise_raw(X_raw_nct: np.ndarray, mode: str, level: float, rng: np.random.Generator) -> np.ndarray:
    Xn = X_raw_nct.astype(np.float32).copy()
    sigma = Xn.transpose(1, 0, 2).reshape(Xn.shape[1], -1).std(axis=1) + 1e-8  # (C,)

    if mode == "gauss":
        noise = rng.normal(0.0, 1.0, size=Xn.shape).astype(np.float32)
        Xn = Xn + noise * (level * sigma[None, :, None])
    elif mode == "bias":
        b = (level * sigma).astype(np.float32)
        Xn = Xn + b[None, :, None]
    elif mode == "scale":
        Xn = Xn * (1.0 + float(level))
    else:
        raise ValueError(f"Unknown sensor noise mode: {mode}")
    return Xn


# ==============================================================================
# 8) Experiment: Train 3 models, then Robustness for EACH model
# ==============================================================================
def run_ucihar_experiment(cfg: Config):
    # ---- Load RAW ----
    X_train_raw, y_train = load_ucihar_split(cfg.data_dir, "train")
    X_test_raw,  y_test  = load_ucihar_split(cfg.data_dir, "test")

    # ---- Train/Val split ----
    idx = np.arange(len(X_train_raw))
    tr_idx, va_idx = train_test_split(idx, test_size=cfg.val_split, random_state=SEED, stratify=y_train)

    # ---- Train-only normalization stats ----
    mean_c, std_c = compute_channel_mean_std(X_train_raw[tr_idx])

    # ---- Normalize clean splits ----
    Xtr = normalize_ntc(X_train_raw[tr_idx], mean_c, std_c)
    ytr = y_train[tr_idx]
    Xva = normalize_ntc(X_train_raw[va_idx], mean_c, std_c)
    yva = y_train[va_idx]
    Xte = normalize_ntc(X_test_raw, mean_c, std_c)
    yte = y_test

    # ---- Loaders ----
    pin = (cfg.device == "cuda")
    g = torch.Generator(device="cpu").manual_seed(SEED)

    train_loader = DataLoader(PreloadedDataset(Xtr, ytr), batch_size=cfg.batch_size, shuffle=True,
                              num_workers=cfg.num_workers, generator=g, pin_memory=pin)
    val_loader   = DataLoader(PreloadedDataset(Xva, yva), batch_size=cfg.batch_size, shuffle=False,
                              num_workers=cfg.num_workers, pin_memory=pin)
    test_loader  = DataLoader(PreloadedDataset(Xte, yte), batch_size=cfg.batch_size, shuffle=False,
                              num_workers=cfg.num_workers, pin_memory=pin)

    # ---- Robust settings (your exact list) ----
    tempo_speeds = [0.85, 0.90, 0.95, 1.05, 1.10, 1.15]
    gauss_levels = [0.05, 0.10, 0.20]
    bias_levels  = [0.05, 0.10, 0.20]
    scale_levels = [0.05, -0.05, 0.10, -0.10, 0.20, -0.20]

    # raw test as (N,C,T)
    X_test_nct = np.transpose(X_test_raw, (0, 2, 1)).astype(np.float32)
    rng = np.random.default_rng(SEED)

    def eval_model_on_perturbed(model, Xpert_nct: np.ndarray) -> Tuple[float, float]:
        Xpert_ntc = np.transpose(Xpert_nct, (0, 2, 1)).astype(np.float32)
        Xnorm = normalize_ntc(Xpert_ntc, mean_c, std_c)
        loader = DataLoader(PreloadedDataset(Xnorm, y_test),
                            batch_size=cfg.batch_size, shuffle=False,
                            num_workers=cfg.num_workers, pin_memory=pin)
        return evaluate(model, loader, cfg)

    # ---- Train + Robust per model ----
    model_names = ["GAP", "TPA", "Gated-TPA"]

    print("\n" + "="*80)
    print("UCI-HAR | CLEAN + ROBUSTNESS (PER MODEL)  |  Backbone: Transformer Encoder")
    print("="*80)

    for mn in model_names:
        set_seed(SEED)
        model = create_model(mn, cfg)

        train_info = train_model(model, train_loader, val_loader, cfg)
        clean_acc, clean_f1 = evaluate(model, test_loader, cfg)

        print("\n" + "-"*80)
        print(f"[{mn}]  Clean Test: Acc={clean_acc:.4f} | F1={clean_f1:.4f} | (Val best F1={train_info['best_val_f1']:.4f})")
        print("-"*80)

        # 1) Tempo
        print("[1) Temporal Scaling]")
        for sp in tempo_speeds:
            Xp = apply_tempo_shift_raw(X_test_nct, sp)
            acc, f1 = eval_model_on_perturbed(model, Xp)
            df1 = clean_f1 - f1
            print(f"speed={sp:>4.2f} | Acc={acc:.4f} | F1={f1:.4f} | ΔF1={df1:+.4f}")

        # 2) Gauss
        print("\n[2) Additive Gaussian Noise]")
        for lv in gauss_levels:
            Xp = apply_sensor_noise_raw(X_test_nct, mode="gauss", level=lv, rng=rng)
            acc, f1 = eval_model_on_perturbed(model, Xp)
            df1 = clean_f1 - f1
            print(f"σ={lv:>4.2f} | Acc={acc:.4f} | F1={f1:.4f} | ΔF1={df1:+.4f}")

        # 3) Bias
        print("\n[3) Additive Bias Drift]")
        for lv in bias_levels:
            Xp = apply_sensor_noise_raw(X_test_nct, mode="bias", level=lv, rng=rng)
            acc, f1 = eval_model_on_perturbed(model, Xp)
            df1 = clean_f1 - f1
            print(f"δ={lv:>4.2f} | Acc={acc:.4f} | F1={f1:.4f} | ΔF1={df1:+.4f}")

        # 4) Scale
        print("\n[4) Multiplicative Scale Drift]")
        for lv in scale_levels:
            Xp = apply_sensor_noise_raw(X_test_nct, mode="scale", level=lv, rng=rng)
            acc, f1 = eval_model_on_perturbed(model, Xp)
            df1 = clean_f1 - f1
            sign = "+" if lv > 0 else ""
            print(f"s={sign}{lv:>4.2f} | Acc={acc:.4f} | F1={f1:.4f} | ΔF1={df1:+.4f}")

    print("\n" + "="*80)
    print("DONE")
    print("="*80)


# ==============================================================================
# 9) Run
# ==============================================================================
if __name__ == "__main__":
    run_ucihar_experiment(cfg)


UCI-HAR | CLEAN + ROBUSTNESS (PER MODEL)  |  Backbone: Transformer Encoder


/tmp/ipython-input-62712/2715642493.py:202: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(



--------------------------------------------------------------------------------
[GAP]  Clean Test: Acc=0.8850 | F1=0.8837 | (Val best F1=0.9742)
--------------------------------------------------------------------------------
[1) Temporal Scaling]
speed=0.85 | Acc=0.8839 | F1=0.8826 | ΔF1=+0.0011
speed=0.90 | Acc=0.8860 | F1=0.8848 | ΔF1=-0.0011
speed=0.95 | Acc=0.8850 | F1=0.8837 | ΔF1=-0.0000
speed=1.05 | Acc=0.8856 | F1=0.8844 | ΔF1=-0.0007
speed=1.10 | Acc=0.8853 | F1=0.8841 | ΔF1=-0.0004
speed=1.15 | Acc=0.8860 | F1=0.8848 | ΔF1=-0.0011

[2) Additive Gaussian Noise]
σ=0.05 | Acc=0.8863 | F1=0.8847 | ΔF1=-0.0010
σ=0.10 | Acc=0.8850 | F1=0.8824 | ΔF1=+0.0013
σ=0.20 | Acc=0.8633 | F1=0.8564 | ΔF1=+0.0273

[3) Additive Bias Drift]
δ=0.05 | Acc=0.8846 | F1=0.8833 | ΔF1=+0.0004
δ=0.10 | Acc=0.8816 | F1=0.8802 | ΔF1=+0.0035
δ=0.20 | Acc=0.8700 | F1=0.8684 | ΔF1=+0.0153

[4) Multiplicative Scale Drift]
s=+0.05 | Acc=0.8738 | F1=0.8726 | ΔF1=+0.0111
s=-0.05 | Acc=0.8901 | F1=0.8886 | ΔF1

/tmp/ipython-input-62712/2715642493.py:202: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(



--------------------------------------------------------------------------------
[TPA]  Clean Test: Acc=0.9006 | F1=0.9007 | (Val best F1=0.9685)
--------------------------------------------------------------------------------
[1) Temporal Scaling]
speed=0.85 | Acc=0.9033 | F1=0.9037 | ΔF1=-0.0030
speed=0.90 | Acc=0.9033 | F1=0.9036 | ΔF1=-0.0029
speed=0.95 | Acc=0.9043 | F1=0.9047 | ΔF1=-0.0040
speed=1.05 | Acc=0.9030 | F1=0.9033 | ΔF1=-0.0026
speed=1.10 | Acc=0.9026 | F1=0.9030 | ΔF1=-0.0023
speed=1.15 | Acc=0.9036 | F1=0.9040 | ΔF1=-0.0033

[2) Additive Gaussian Noise]
σ=0.05 | Acc=0.9006 | F1=0.9007 | ΔF1=-0.0000
σ=0.10 | Acc=0.8996 | F1=0.8994 | ΔF1=+0.0013
σ=0.20 | Acc=0.8996 | F1=0.8993 | ΔF1=+0.0014

[3) Additive Bias Drift]
δ=0.05 | Acc=0.8979 | F1=0.8979 | ΔF1=+0.0028
δ=0.10 | Acc=0.8972 | F1=0.8970 | ΔF1=+0.0037
δ=0.20 | Acc=0.8711 | F1=0.8714 | ΔF1=+0.0293

[4) Multiplicative Scale Drift]
s=+0.05 | Acc=0.9040 | F1=0.9040 | ΔF1=-0.0033
s=-0.05 | Acc=0.9026 | F1=0.9026 | ΔF1

/tmp/ipython-input-62712/2715642493.py:202: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(



--------------------------------------------------------------------------------
[Gated-TPA]  Clean Test: Acc=0.9060 | F1=0.9063 | (Val best F1=0.9679)
--------------------------------------------------------------------------------
[1) Temporal Scaling]
speed=0.85 | Acc=0.9060 | F1=0.9063 | ΔF1=-0.0001
speed=0.90 | Acc=0.9070 | F1=0.9074 | ΔF1=-0.0011
speed=0.95 | Acc=0.9070 | F1=0.9074 | ΔF1=-0.0011
speed=1.05 | Acc=0.9074 | F1=0.9078 | ΔF1=-0.0015
speed=1.10 | Acc=0.9077 | F1=0.9081 | ΔF1=-0.0018
speed=1.15 | Acc=0.9080 | F1=0.9085 | ΔF1=-0.0022

[2) Additive Gaussian Noise]
σ=0.05 | Acc=0.9070 | F1=0.9073 | ΔF1=-0.0010
σ=0.10 | Acc=0.9063 | F1=0.9065 | ΔF1=-0.0002
σ=0.20 | Acc=0.9084 | F1=0.9083 | ΔF1=-0.0020

[3) Additive Bias Drift]
δ=0.05 | Acc=0.9019 | F1=0.9024 | ΔF1=+0.0039
δ=0.10 | Acc=0.8985 | F1=0.8990 | ΔF1=+0.0073
δ=0.20 | Acc=0.8962 | F1=0.8960 | ΔF1=+0.0103

[4) Multiplicative Scale Drift]
s=+0.05 | Acc=0.9050 | F1=0.9050 | ΔF1=+0.0013
s=-0.05 | Acc=0.9087 | F1=0.9091